
# Phase IV — ArtBench-10 style generalization

This pilot asks whether the multiscale level-set geometry that discriminates **artists** also carries information about **artistic style**, and whether that signal generalizes to **artists never seen during training**.

The primary scientific test is not the ordinary ArtBench image split. It is:

\[
\boxed{\text{artist-disjoint style classification}}
\]

with the artist treated as a grouping variable in nested cross-validation.

We compare the same conceptual families used earlier:

- **B**: strong conventional appearance baseline;
- **G**: multiscale level-set geometry + structure-tensor summaries;
- **B+G**: combined representation;
- dimension-matched \(k=40\) versions.

Two analyses are reported:

1. **ArtBench-10 all 10 styles**;
2. **WikiArt-derived 8-style sensitivity analysis**, excluding `surrealism` and `ukiyo_e`, because ArtBench documents those two classes as coming from single-style external databases rather than WikiArt. This controls an important source/digitization confound.

This is a **pilot**: 300 training + 100 test images per style by default. If the artist-disjoint gain survives, the next run can scale to the full corpus.


In [ ]:

import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())



## 1. Download only the 256×256 ArtBench-10 split and official metadata

The official ArtBench repository provides 60,000 images, balanced across 10 styles: 5,000 train and 1,000 test per class. We use the 256×256 ImageFolder split so that the pilot does not download the much larger original-resolution archives.

The cell first tries the Kaggle dataset `alexanderliao/artbench10` using single-file download. If the internal Kaggle path differs, it falls back to the official ArtBench file URL.


In [ ]:

import tarfile, urllib.request, pandas as pd, numpy as np
import kagglehub

DATA_DIR = Path("/content/artbench_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_HANDLE = "alexanderliao/artbench10"
TAR_NAME = "artbench-10-imagefolder-split.tar"
tar_path = None

# Avoid accidentally downloading the entire original-resolution archive collection.
candidate_paths = [
    TAR_NAME,
    f"256X256/{TAR_NAME}",
    f"data/256X256/{TAR_NAME}",
    f"ArtBench-10/data/256X256/{TAR_NAME}",
]

for candidate in candidate_paths:
    try:
        print("Trying Kaggle file:", candidate)
        p = Path(kagglehub.dataset_download(KAGGLE_HANDLE, path=candidate))
        if p.exists():
            tar_path = p
            print("Downloaded from Kaggle:", p)
            break
    except Exception as exc:
        print("  not found:", type(exc).__name__)

if tar_path is None:
    tar_path = DATA_DIR / TAR_NAME
    official = "https://artbench.eecs.berkeley.edu/files/artbench-10-imagefolder-split.tar"
    print("Falling back to official ArtBench URL (~1.85 GB)...")
    subprocess.run(["wget", "-c", official, "-O", str(tar_path)], check=True)

EXTRACT_DIR = DATA_DIR / "imagefolder"
if not any(EXTRACT_DIR.rglob("train")):
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print("Extracting:", tar_path)
    with tarfile.open(tar_path) as tf:
        try:
            tf.extractall(EXTRACT_DIR, filter="data")
        except TypeError:
            tf.extractall(EXTRACT_DIR)

# Official metadata; needed for artist-disjoint evaluation.
METADATA_CSV = DATA_DIR / "ArtBench-10.csv"
if not METADATA_CSV.exists():
    metadata_url = "https://artbench.eecs.berkeley.edu/files/ArtBench-10.csv"
    try:
        subprocess.run(["wget", "-q", metadata_url, "-O", str(METADATA_CSV)], check=True)
    except Exception:
        print("Official metadata download failed; upload ArtBench-10.csv manually.")
        from google.colab import files
        uploaded = files.upload()
        if "ArtBench-10.csv" not in uploaded:
            raise FileNotFoundError("ArtBench-10.csv is required for the artist-disjoint experiment.")
        METADATA_CSV.write_bytes(uploaded["ArtBench-10.csv"])

print("ImageFolder root:", EXTRACT_DIR)
print("Metadata:", METADATA_CSV, "size:", METADATA_CSV.stat().st_size)
print("Metadata columns:", pd.read_csv(METADATA_CSV, nrows=3).columns.tolist())



## 2. Build a manifest and audit the artist metadata

The mapping is intentionally checked before any model is fit. We require high metadata coverage for the artist-disjoint experiment. The ordinary image-split benchmark can still run if metadata matching fails, but it is not the central result.


In [ ]:

RESULTS = REPO_DIR / "results" / "phase4_artbench"
RESULTS.mkdir(parents=True, exist_ok=True)

MANIFEST = RESULTS / "artbench_manifest.csv"

subprocess.run(
    [
        sys.executable, "-u", "scripts/prepare_artbench_manifest.py",
        "--dataset-root", str(EXTRACT_DIR),
        "--metadata-csv", str(METADATA_CSV),
        "--output", str(MANIFEST),
    ],
    check=True,
)

manifest = pd.read_csv(MANIFEST)
display(manifest.head())
print("\nCounts:")
display(manifest.groupby(["split", "style"]).size().unstack(fill_value=0))

coverage = manifest["metadata_match"].mean()
print(f"Metadata match coverage: {coverage:.2%}")
if coverage >= 0.90:
    artist_counts = (
        manifest[manifest["artist"].fillna("").astype(str).str.strip().ne("")]
        .groupby("style")["artist"].nunique()
        .sort_values()
    )
    print("\nUnique artists by style:")
    display(artist_counts.to_frame("n_artists"))
else:
    print("WARNING: artist-disjoint evaluation will be skipped unless the filename↔metadata mapping is fixed.")



## 3. Pilot feature extraction

Default pilot size:

\[
300\ \text{train/style}+100\ \text{test/style}=4{,}000\ \text{paintings}.
\]

Sampling caps the contribution from a single artist when artist metadata are available.

At native 256 px, the geometric scales are defined relative to the 512 px reference used in Phase III:

\[
\sigma_{\rm px}=\sigma_{\rm ref}\frac{256}{512},
\qquad
\sigma_{\rm ref}\in\{1,2,4,8\}.
\]

Geometry uses the derivative-of-Gaussian, scale-normalized Phase-III implementation. The baseline contains multiscale gradients, edge densities, orientation histograms, multi-distance GLCM, and LBP.


In [ ]:

FEATURES = RESULTS / "artbench_pilot_features.csv"

PILOT_TRAIN_PER_STYLE = 300
PILOT_TEST_PER_STYLE = 100
MAX_TRAIN_PER_ARTIST = 40
MAX_TEST_PER_ARTIST = 20
RECOMPUTE_FEATURES = True

if RECOMPUTE_FEATURES or not FEATURES.exists():
    subprocess.run(
        [
            sys.executable, "-u", "scripts/extract_artbench_style_features.py",
            "--manifest", str(MANIFEST),
            "--output", str(FEATURES),
            "--train-per-style", str(PILOT_TRAIN_PER_STYLE),
            "--test-per-style", str(PILOT_TEST_PER_STYLE),
            "--max-train-per-artist", str(MAX_TRAIN_PER_ARTIST),
            "--max-test-per-artist", str(MAX_TEST_PER_ARTIST),
            "--long-side", "256",
            "--sigma-refs", "1", "2", "4", "8",
            "--reference-long-side", "512",
            "--seed", "42",
        ],
        check=True,
    )

feat = pd.read_csv(FEATURES)
print("Feature matrix:", feat.shape)
print("Baseline:", sum(c.startswith("base__") for c in feat.columns))
print("Geometry:", sum(c.startswith("geom__") for c in feat.columns))
display(feat.groupby(["split", "style"]).size().unstack(fill_value=0))

artist_coverage = feat["artist"].fillna("").astype(str).str.strip().ne("").mean()
print(f"Artist coverage in pilot: {artist_coverage:.2%}")



## 4. Style classification: ordinary split vs artist-disjoint generalization

The ordinary ArtBench split is reported for comparability, but it may contain the same artist on both sides.

The primary experiment uses nested **StratifiedGroupKFold**:

- outer folds: completely unseen artists;
- inner folds: training-only hyperparameter selection;
- grouping variable: artist;
- metric: Macro-F1;
- uncertainty: artist-level group bootstrap.

We also repeat the analysis after removing `surrealism` and `ukiyo_e`. The ArtBench paper states that those two classes come from single-style external sources, whereas the other eight are WikiArt-derived; therefore the 8-style analysis is a source-confound sensitivity test.


In [ ]:

EXP_DIR = RESULTS / "experiments"
EXP_DIR.mkdir(exist_ok=True)

subprocess.run(
    [
        sys.executable, "-u", "scripts/run_artbench_style_generalization.py",
        "--features", str(FEATURES),
        "--output-dir", str(EXP_DIR),
        "--matched-k", "40",
        "--cv-folds", "3",
        "--outer-folds", "5",
        "--inner-folds", "3",
        "--n-jobs", "-1",
    ],
    check=True,
)

def read_if(name):
    p = EXP_DIR / name
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

official = read_if("artbench_official_results.csv")
official_d = read_if("artbench_official_deltas.csv")
grouped = read_if("artbench_artist_disjoint_results.csv")
grouped_d = read_if("artbench_artist_disjoint_deltas.csv")
folds = read_if("artbench_artist_disjoint_fold_results.csv")

print("OFFICIAL IMAGE SPLIT")
display(official)
display(official_d)

print("\nARTIST-DISJOINT NESTED CV")
display(grouped)
display(grouped_d)



## 5. Decision rule for the pilot

The **primary GO criterion** is deliberately stringent:

\[
\Delta F_1 =
F_1(B+G)-F_1(B)>0
\]

with the **95% artist-group bootstrap interval entirely above zero** in artist-disjoint evaluation.

The strongest result would be if this holds for both:

1. all ten styles;
2. the WikiArt-derived 8-style sensitivity subset.

The matched-\(k=40\) contrast is a second protection against a pure dimensionality advantage.


In [ ]:

if grouped_d.empty:
    print("Artist-disjoint results unavailable. Inspect metadata mapping before interpreting style generalization.")
else:
    key = grouped_d[
        (grouped_d["new_model"] == "BG_combined_full")
        & (grouped_d["reference"] == "B_strong_full")
    ].copy()
    key["GO"] = key["delta_ci_low"] > 0
    display(
        key[
            [
                "dataset", "delta_macro_f1", "delta_ci_low", "delta_ci_high",
                "bootstrap_p_improvement", "n_images", "n_artists", "GO"
            ]
        ]
    )

    print("\nMatched-dimensionality contrasts:")
    display(
        grouped_d[
            grouped_d["new_model"].str.contains("k40", na=False)
        ][
            [
                "dataset", "new_model", "reference",
                "delta_macro_f1", "delta_ci_low", "delta_ci_high",
                "bootstrap_p_improvement"
            ]
        ]
    )


In [ ]:

import matplotlib.pyplot as plt

if not grouped.empty:
    plot_df = grouped[
        grouped["experiment"].isin(["B_strong_full", "G_geometry_full", "BG_combined_full"])
    ].copy()

    for dataset_name, sub in plot_df.groupby("dataset"):
        sub = sub.sort_values("experiment")
        x = np.arange(len(sub))
        plt.figure(figsize=(7, 4))
        plt.errorbar(
            x,
            sub["macro_f1_oof"],
            yerr=[
                sub["macro_f1_oof"] - sub["macro_f1_group_boot_ci_low"],
                sub["macro_f1_group_boot_ci_high"] - sub["macro_f1_oof"],
            ],
            fmt="o",
            capsize=4,
        )
        plt.xticks(x, sub["experiment"], rotation=25, ha="right")
        plt.ylabel("Artist-disjoint OOF Macro-F1")
        plt.title(dataset_name)
        plt.tight_layout()
        out = RESULTS / f"Figure_{dataset_name}_artist_disjoint_macroF1.png"
        plt.savefig(out, dpi=220, bbox_inches="tight")
        plt.show()



## 6. What not to claim yet

A positive pilot would support the statement that level-set geometry contains style information that generalizes across unseen artists.

It would **not yet** establish a universal theory of artistic style. Before the manuscript is frozen we would then decide whether to:

- scale from the 4,000-image pilot toward the full ArtBench corpus;
- run the style-level scale anatomy to test the hypothesized hierarchy:
  fine scales → artist discrimination, broader scales → movement/style structure;
- compare the result explicitly with the completed artist-level experiment.


In [ ]:

# Package all lightweight outputs plus the pilot feature matrix for review.
from google.colab import files
import json, shutil

PACKAGE = Path("/content/painting_geometry_phase4_artbench_pilot")
if PACKAGE.exists():
    shutil.rmtree(PACKAGE)
PACKAGE.mkdir()

for p in RESULTS.rglob("*"):
    if p.is_file():
        rel = p.relative_to(RESULTS)
        dst = PACKAGE / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dst)

meta = {
    "branch": BRANCH,
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "dataset": "ArtBench-10",
    "kaggle_handle": KAGGLE_HANDLE,
    "pilot_train_per_style": PILOT_TRAIN_PER_STYLE,
    "pilot_test_per_style": PILOT_TEST_PER_STYLE,
    "long_side": 256,
    "sigma_refs_at_512": [1, 2, 4, 8],
    "primary_protocol": "artist-disjoint nested CV",
    "source_sensitivity": "exclude surrealism and ukiyo_e",
}
(PACKAGE / "phase4_run_metadata.json").write_text(json.dumps(meta, indent=2))

zip_path = shutil.make_archive(
    "/content/painting_geometry_phase4_artbench_pilot",
    "zip",
    root_dir=PACKAGE,
)
print("Created:", zip_path)
files.download(zip_path)
